## Mạng Nơ-ron Đồ thị (GNN) trên Tập dữ liệu ACM

Notebook Colab này trình bày cách triển khai và huấn luyện Mạng Nơ-ron Đồ thị (GNN), đặc biệt là GTN (Graph Transformer Networks) và FastGTN, trên tập dữ liệu ACM (Association for Computing Machinery). Tập dữ liệu ACM là một tập dữ liệu đồ thị dị thể thường được sử dụng cho các tác vụ phân loại nút.

Quá trình này bao gồm:
1.  **Thiết lập môi trường**: Cài đặt các thư viện PyTorch Geometric cần thiết.
2.  **Tải và Chuẩn bị dữ liệu**: Tải file `ACM.mat`, chứa cấu trúc đồ thị và các đặc trưng nút, sau đó tổ chức lại dữ liệu.
3.  **Kỹ thuật đặc trưng (Feature Engineering)**: Tạo các đặc trưng nút và ma trận kề cho các loại quan hệ khác nhau.
4.  **Chia tách dữ liệu**: Chia tập dữ liệu thành các tập huấn luyện, kiểm định và kiểm tra.
5.  **Huấn luyện mô hình**: Triển khai và huấn luyện các mô hình GTN và FastGTN.
6.  **Đánh giá**: Đánh giá hiệu suất mô hình bằng các chỉ số như F1-score.

### 1. Cài đặt Thư viện

Bước này cài đặt các thư viện cần thiết cho việc xử lý đồ thị với PyTorch Geometric (torch-scatter, torch-sparse, torch-cluster, torch-spline-conv, torch-geometric) và các thư viện khoa học dữ liệu chung như `scipy` và `scikit-learn`. Các thư viện `torch-geometric` được cài đặt để tương thích với phiên bản PyTorch và CUDA hiện có trong môi trường Colab.

In [ ]:
import torch
import os

# 1. Cài đặt các thư viện đồ thị tương thích với phiên bản Torch của Colab
v = torch.__version__.split('+')[0]
c = torch.version.cuda.replace('.', '')
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-{v}+cu{c}.html
!pip install scipy scikit-learn

Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html


### 2. Chuẩn bị Dữ liệu ACM

Bước này đảm bảo rằng file `ACM.mat` (chứa dữ liệu đồ thị) được đặt trong thư mục `data/`. Nếu thư mục `data` chưa tồn tại, nó sẽ được tạo. Sau đó, file `ACM.mat` sẽ được sao chép vào thư mục này để các bước tiếp theo có thể truy cập dữ liệu dễ dàng.

In [ ]:
import os
import shutil

# Tạo thư mục data nếu chưa có
if not os.path.exists('data'):
    os.makedirs('data')

# Tìm file ACM.mat bạn đã upload và đưa nó vào thư mục data
if os.path.exists('ACM.mat'):
    shutil.copy('ACM.mat', 'data/ACM.mat')
    print("Đã đưa ACM.mat vào thư mục data/. Sẵn sàng chạy!")
else:
    # Nếu bạn chưa upload, lệnh này sẽ báo để bạn biết
    print("LỖI: Bạn chưa upload file ACM.mat lên Colab. Hãy nhấn nút upload bên trái!")

Đã đưa ACM.mat vào thư mục data/. Sẵn sàng chạy!


### 3. Tải Dữ liệu Đồ thị

Sử dụng thư viện `scipy.io` để tải file `ACM.mat` vào biến `mat_file`. File `.mat` này là một định dạng dữ liệu thường được sử dụng trong MATLAB và chứa cấu trúc của đồ thị ACM.

In [ ]:
from scipy import io
import numpy as np
from scipy.sparse import csr_matrix
mat_file = io.loadmat('data/ACM.mat')

### 4. Khám phá Cấu trúc Dữ liệu `mat_file`

Cell này hiển thị nội dung của `mat_file`. Chúng ta có thể thấy nó chứa nhiều ma trận thưa (`COOrdinate sparse matrix`) đại diện cho các mối quan hệ khác nhau giữa các thực thể (như Paper-Venue, Paper-Author, Paper-Subject) và các mảng (`array`) chứa tên của các Author (A), Conference (C), Faculty (F), Subject (L), Term (T), và Venue (V).

**Các thành phần chính:**
-   `TvsP`: Ma trận thưa biểu diễn mối quan hệ giữa Terms và Papers.
-   `PvsA`: Ma trận thưa biểu diễn mối quan hệ giữa Papers và Authors.
-   `PvsV`: Ma trận thưa biểu diễn mối quan hệ giữa Papers và Venues.
-   `AvsF`: Ma trận thưa biểu diễn mối quan hệ giữa Authors và Faculties.
-   `VvsC`: Ma trận thưa biểu diễn mối quan hệ giữa Venues và Conferences.
-   `PvsL`: Ma trận thưa biểu diễn mối quan hệ giữa Papers và Subjects.
-   `PvsC`: Ma trận thưa biểu diễn mối quan hệ giữa Papers và Conferences (dùng để xác định nhãn).
-   `A`, `C`, `F`, `L`, `T`, `V`: Các mảng chứa tên (metadata) của các thực thể tương ứng.

In [ ]:
mat_file

{'__header__': b'MATLAB 5.0 MAT-file, Platform: PCWIN64, Created on: Mon Aug 08 18:23:50 2011',
 '__version__': '1.0',
 '__globals__': [],
 'TvsP': <COOrdinate sparse matrix of dtype 'float64'
 	with 972973 stored elements and shape (1903, 12499)>,
 'PvsA': <COOrdinate sparse matrix of dtype 'float64'
 	with 37055 stored elements and shape (12499, 17431)>,
 'PvsV': <COOrdinate sparse matrix of dtype 'float64'
 	with 12499 stored elements and shape (12499, 196)>,
 'AvsF': <COOrdinate sparse matrix of dtype 'float64'
 	with 30424 stored elements and shape (17431, 1804)>,
 'VvsC': <COOrdinate sparse matrix of dtype 'float64'
 	with 196 stored elements and shape (196, 14)>,
 'PvsL': <COOrdinate sparse matrix of dtype 'float64'
 	with 12499 stored elements and shape (12499, 73)>,
 'PvsC': <COOrdinate sparse matrix of dtype 'float64'
 	with 12499 stored elements and shape (12499, 14)>,
 'A': array([[array(['Raluca Paiu'], dtype='<U11')],
        [array(['Panagiotis Karras'], dtype='<U17')],


### 5. Lấy và Gán Nhãn cho Giấy Tờ (Papers)

Trong bước này, chúng ta trích xuất và gán nhãn (target) cho các bài báo dựa trên các hội nghị mà chúng xuất hiện. Cụ thể:
-   `paper_conf`: Lấy chỉ số của các hội nghị mà mỗi bài báo thuộc về từ ma trận `PvsC`.
-   `paper_db`, `paper_dm`, `paper_wc`: Xác định các bài báo thuộc ba danh mục chính: Database (DB), Data Mining (DM), và Wireless Communication (WC) dựa trên các chỉ số hội nghị đã cho ([1, 13] cho DB, [0] cho DM, [9, 10] cho WC).
-   `paper_db_idx`: Chọn ngẫu nhiên 994 bài báo từ danh mục Database để đảm bảo số lượng cân bằng hơn.
-   `paper_idx`: Tổng hợp tất cả các chỉ số bài báo đã chọn từ ba danh mục.
-   `paper_target`: Gán nhãn số (0 cho DB, 1 cho WC, 2 cho DM) cho từng bài báo trong `paper_idx`.

In [ ]:
paper_conf = mat_file['PvsC'].nonzero()[1]

In [ ]:
# DataBase
paper_db = np.isin(paper_conf,[1,13])
paper_db_idx = np.where(paper_db == True)[0]
paper_db_idx = np.sort(np.random.choice(paper_db_idx,994,replace=False))
# Data Mining
paper_dm = np.isin(paper_conf,[0])
paper_dm_idx = np.where(paper_dm == True)[0]
# Wireless Communication
paper_wc = np.isin(paper_conf,[9,10])
paper_wc_idx = np.where(paper_wc == True)[0]

In [ ]:
paper_idx = np.sort(list(paper_db_idx)+list(paper_dm_idx)+list(paper_wc_idx))

In [ ]:
len(paper_idx)

3025

In [ ]:
# 0 : database, 1: wireless communication, 2: data mining
paper_target = []
for idx in paper_idx:
    if idx in paper_db_idx:
        paper_target.append(0)
    elif idx in paper_wc_idx:
        paper_target.append(1)
    else:
        paper_target.append(2)
paper_target = np.array(paper_target)

In [ ]:
paper_target.shape

(3025,)

### 6. Xử lý các cạnh loại Paper-Author (PAP)

Bước này tập trung vào việc xử lý các mối quan hệ giữa bài báo và tác giả (`PvsA`):
-   `PvsA_csr`: Chuyển đổi ma trận `PvsA` sang định dạng CSR (Compressed Sparse Row) để truy cập hiệu quả hơn.
-   `authors`: Trích xuất các tác giả liên quan đến các bài báo đã chọn (`paper_idx`).
-   `author_dic`: Tạo một từ điển để ánh xạ các chỉ số tác giả gốc sang các chỉ số mới, duy nhất, bắt đầu sau `len(paper_idx)` để tránh xung đột với chỉ số bài báo.
-   `re_authors`: Danh sách các chỉ số tác giả đã được ánh xạ lại.

In [ ]:
PvsA_csr = mat_file['PvsA'].tocsr()
authors = PvsA_csr[paper_idx].nonzero()[1]
author_dic = {}
re_authors = []
for author in authors:
    if author not in author_dic:
        author_dic[author] = len(author_dic) + len(paper_idx)
    re_authors.append(author_dic[author])
re_authors = np.array(re_authors)

In [ ]:
len(author_dic)

6085

### 7. Xử lý các cạnh loại Paper-Subject (PSP)

Tương tự như bước trên, bước này xử lý các mối quan hệ giữa bài báo và chủ đề (`PvsL`):
-   `PvsL_csr`: Chuyển đổi ma trận `PvsL` sang định dạng CSR.
-   `subjects`: Trích xuất các chủ đề liên quan đến các bài báo đã chọn.
-   `subject_dic`: Tạo một từ điển để ánh xạ các chỉ số chủ đề gốc sang các chỉ số mới, duy nhất, bắt đầu sau chỉ số của `paper_idx` và `author_dic`.
-   `re_subjects`: Danh sách các chỉ số chủ đề đã được ánh xạ lại.

In [ ]:
PvsL_csr = mat_file['PvsL'].tocsr()
subjects = PvsL_csr[paper_idx].nonzero()[1]
subject_dic = {}
re_subjects = []
for subject in subjects:
    if subject not in subject_dic:
        subject_dic[subject] = len(subject_dic) + len(paper_idx) + len(author_dic)
    re_subjects.append(subject_dic[subject])
re_subjects = np.array(re_subjects)

In [ ]:
len(subject_dic)

62

### 8. Tính tổng số nút (nodes)

`node_num` là tổng số nút trong đồ thị con được xây dựng, bao gồm:
-   Số lượng bài báo (`len(paper_idx)`)
-   Số lượng tác giả duy nhất (`len(author_dic)`)
-   Số lượng chủ đề duy nhất (`len(subject_dic)`)

Đây là tổng số nút sẽ được sử dụng trong các ma trận kề.

In [ ]:
node_num = len(paper_idx) + len(author_dic) + len(subject_dic)

In [ ]:
node_num

9172

### 9. Xây dựng Ma trận Kề cho các mối quan hệ

Chúng ta tạo ra các ma trận kề thưa (`csr_matrix`) cho các mối quan hệ chính:
-   `A_pa`: Ma trận kề cho mối quan hệ Paper-Author, nơi hàng là Paper và cột là Author. `data` là mảng các giá trị (1 trong trường hợp này, biểu thị sự tồn tại của cạnh).
-   `A_ps`: Ma trận kề cho mối quan hệ Paper-Subject, nơi hàng là Paper và cột là Subject. `data` tương tự như trên.

In [ ]:
papers_pa = PvsA_csr[paper_idx].nonzero()[0]
data_pa = np.ones_like(papers_pa)
A_pa = csr_matrix((data_pa, (papers_pa, re_authors)), shape=(node_num,node_num))

In [ ]:
A_pa

<Compressed Sparse Row sparse matrix of dtype 'int32'
	with 9065 stored elements and shape (9172, 9172)>

In [ ]:
papers_ps = PvsL_csr[paper_idx].nonzero()[0]
data_ps = np.ones_like(papers_ps)
A_ps = csr_matrix((data_ps, (papers_ps, re_subjects)), shape=(node_num,node_num))

In [ ]:
A_ps

<Compressed Sparse Row sparse matrix of dtype 'int32'
	with 3025 stored elements and shape (9172, 9172)>

### 10. Tạo Ma trận Kề Chuyển vị

Để biểu diễn các mối quan hệ hai chiều trong đồ thị dị thể, chúng ta cần các ma trận kề chuyển vị:
-   `A_ap`: Chuyển vị của `A_pa`, biểu diễn mối quan hệ Author-Paper.
-   `A_sp`: Chuyển vị của `A_ps`, biểu diễn mối quan hệ Subject-Paper.

Các ma trận này sẽ được sử dụng làm các loại cạnh (edge types) khác nhau trong mô hình GNN.

In [ ]:
A_ap = A_pa.transpose()

In [ ]:
A_sp = A_ps.transpose()

### 11. Tập hợp các Cạnh (Edges)

`edges` là một danh sách chứa tất cả các ma trận kề đã được xây dựng:
-   `A_pa`: Paper -> Author
-   `A_ap`: Author -> Paper
-   `A_ps`: Paper -> Subject
-   `A_sp`: Subject -> Paper

Danh sách này đại diện cho các loại mối quan hệ khác nhau trong đồ thị dị thể của chúng ta và sẽ được truyền vào mô hình GNN.

In [ ]:
edges = [A_pa,A_ap,A_ps,A_sp]

### 12. Xây dựng Đặc trưng Nút từ Terms (Từ khóa)

Bước này tạo ra các đặc trưng cho các nút (Papers, Authors, Subjects) dựa trên các từ khóa (terms) liên quan đến các bài báo:
-   `terms`: Trích xuất các chỉ số từ khóa từ ma trận `TvsP` (Terms vs Papers) cho các bài báo đã chọn.
-   `term_dic`: Ánh xạ các chỉ số từ khóa gốc sang các chỉ số mới, duy nhất, bắt đầu sau các chỉ số của Paper, Author, Subject.
-   `re_terms`: Danh sách các chỉ số từ khóa đã được ánh xạ lại.

In [ ]:
terms = mat_file['TvsP'].transpose().tocsr()[paper_idx].nonzero()[1]
term_dic = {}
re_terms = []
for term in terms:
    if term not in term_dic:
        term_dic[term] = len(term_dic) + len(paper_idx) + len(author_dic) + len(subject_dic)
    re_terms.append(term_dic[term])
re_terms = np.array(re_terms)

### 13. Kết hợp Đặc trưng Nút

Ở bước này, các đặc trưng cho từng loại nút (paper, author, subject) được tạo ra dựa trên mối quan hệ với các từ khóa, sau đó được kết hợp lại thành một ma trận đặc trưng lớn cho tất cả các nút:

-   `tmp_num_node`: Tổng số nút tạm thời, bao gồm cả các từ khóa.
-   `A_pa_tmp`, `A_ps_tmp`, `A_pt_tmp`: Các ma trận kề tạm thời được xây dựng với `tmp_num_node` để trích xuất đặc trưng.
-   `paper_feat`: Đặc trưng cho các bài báo, được lấy từ ma trận `A_pt_tmp` (Paper -> Term).
-   `author_feat`: Đặc trưng cho các tác giả, được tính bằng cách lan truyền đặc trưng từ khóa qua mối quan hệ Author-Paper-Term.
-   `subject_feat`: Đặc trưng cho các chủ đề, tương tự, được lan truyền qua mối quan hệ Subject-Paper-Term.
-   `node_faeture`: Ma trận đặc trưng cuối cùng, được tạo bằng cách nối `paper_feat`, `author_feat`, và `subject_feat` theo chiều dọc. Kích thước của `node_faeture` là `(tổng_số_nút, số_lượng_từ_khóa_duy_nhất)`.

In [ ]:
mat_file['TvsP'].transpose()

<COOrdinate sparse matrix of dtype 'float64'
	with 972973 stored elements and shape (12499, 1903)>

In [ ]:
# tmp
tmp_num_node = node_num + len(term_dic)
papers = PvsA_csr[paper_idx].nonzero()[0]
data = np.ones_like(papers)
A_pa_tmp = csr_matrix((data, (papers, re_authors)), shape=(tmp_num_node,tmp_num_node))
papers = PvsL_csr[paper_idx].nonzero()[0]
data = np.ones_like(papers)
A_ps_tmp = csr_matrix((data, (papers, re_subjects)), shape=(tmp_num_node,tmp_num_node))
# mat_file['TvsP'] is (1903, 12499), transpose() is (12499, 1903) which matches Pvs*
PvsT_csr = mat_file['TvsP'].transpose().tocsr()
papers = PvsT_csr[paper_idx].nonzero()[0]
data = np.ones_like(papers)
A_pt_tmp = csr_matrix((data, (papers, re_terms)), shape=(tmp_num_node,tmp_num_node))

In [ ]:
paper_feat = np.array(A_pt_tmp[:len(paper_idx),-len(term_dic):].toarray()>0, dtype=int)
author_feat = np.array(A_pa_tmp.transpose().dot(A_pt_tmp)[len(paper_idx):len(paper_idx)+len(author_dic),-len(term_dic):].toarray()>0, dtype=int)
subject_feat = np.array(A_ps_tmp.transpose().dot(A_pt_tmp)[len(paper_idx)+len(author_dic):len(paper_idx)+len(author_dic)+len(subject_dic),-len(term_dic):].toarray()>0, dtype=int)

In [ ]:
node_faeture = np.concatenate((paper_feat,author_feat,subject_feat))

In [ ]:
node_faeture.shape

(9172, 1903)

In [ ]:
paper_target.shape

(3025,)

### 14. Chia tập dữ liệu thành Train/Validation/Test

Bước này chuẩn bị các nhãn và chia dữ liệu thành các tập huấn luyện, validation và kiểm tra:

-   `train_valid_DB`, `train_valid_WC`, `train_valid_DM`: Chọn ngẫu nhiên các chỉ số bài báo từ mỗi danh mục (Database, Wireless Communication, Data Mining) để tạo thành tập huấn luyện và validation.
-   `train_idx`, `valid_idx`: Tổng hợp các chỉ số bài báo cho tập huấn luyện và validation.
-   `train_target`, `valid_target`: Lấy nhãn tương ứng cho các bài báo trong tập huấn luyện và validation.
-   `train_label`, `valid_label`: Kết hợp chỉ số và nhãn thành các cặp `[index, label]` cho tập huấn luyện và validation.
-   `test_idx`, `test_target`, `test_label`: Xác định các bài báo còn lại (không có trong tập huấn luyện và validation) làm tập kiểm tra và tạo `test_label`.

`labels` là một danh sách chứa `train_label`, `valid_label`, và `test_label`.

In [ ]:
# Train, Valid
train_valid_DB = list(np.random.choice(np.where(paper_target==0)[0],300, replace=False))
train_valid_WC = list(np.random.choice(np.where(paper_target==1)[0],300, replace=False))
train_valid_DM = list(np.random.choice(np.where(paper_target==2)[0],300, replace=False))

train_idx = np.array(train_valid_DB[:200] + train_valid_WC[:200] + train_valid_DM[:200])
train_target = paper_target[train_idx]
train_label = np.vstack((train_idx,train_target)).transpose()
valid_idx = np.array(train_valid_DB[200:] + train_valid_WC[200:] + train_valid_DM[200:])
valid_target = paper_target[valid_idx]
valid_label = np.vstack((valid_idx,valid_target)).transpose()
test_idx = np.array(list((set(np.arange(paper_target.shape[0])) - set(train_idx)) - set(valid_idx)))
test_target = paper_target[test_idx]
test_label = np.vstack((test_idx,test_target)).transpose()

In [ ]:
labels = [train_label,valid_label,test_label]

In [ ]:
labels

[array([[1328,    0],
        [3017,    0],
        [1564,    0],
        ...,
        [  11,    2],
        [ 224,    2],
        [ 641,    2]]),
 array([[1308,    0],
        [1525,    0],
        [3000,    0],
        [1216,    0],
        [1697,    0],
        [2928,    0],
        [1439,    0],
        [1133,    0],
        [1267,    0],
        [3001,    0],
        [1314,    0],
        [1683,    0],
        [1084,    0],
        [2756,    0],
        [1401,    0],
        [1536,    0],
        [1404,    0],
        [2771,    0],
        [1219,    0],
        [1109,    0],
        [2809,    0],
        [1619,    0],
        [1126,    0],
        [1632,    0],
        [1665,    0],
        [1097,    0],
        [1467,    0],
        [2861,    0],
        [2821,    0],
        [3013,    0],
        [1367,    0],
        [1554,    0],
        [1592,    0],
        [1435,    0],
        [2827,    0],
        [1353,    0],
        [2993,    0],
        [1122,    0],
        [1300,   

### 15. Chuẩn bị các file Python cho mô hình

Các `main_gpu.py` được gọi ở dưới đây đã thất bại vì thiếu các file `model_gtn.py`, `model_fastgtn.py`, `layer.py`, `layer_fastgtn.py`, `inits.py` và `utils.py`. Các cell tiếp theo sẽ tạo ra các file này.

`main_gpu.py` (mà chúng ta đang chạy thông qua `2HotfwsnRJFo`) và `main.py` là các script chính để chạy các mô hình GTN và FastGTN. Chúng đọc các đối số dòng lệnh để cấu hình mô hình (ví dụ: dataset, model type, epochs, learning rate, number of layers, v.v.).

-   **Mô hình GTN**: Được gọi với các đối số cơ bản như `num_layers=1`, `epoch=100`, `lr=0.02`, `num_channels=2`.
-   **Mô hình FastGTN**: Được gọi với các đối số nâng cao hơn như `num_layers=3`, `epoch=200`, `lr=0.05`, `channel_agg=mean`, `num_channels=2`, `non_local_weight=-1`, `K=1`, `non_local`.

Các lệnh này cố gắng chạy script huấn luyện mô hình với các cấu hình khác nhau cho GTN và FastGTN. Output `python3: can't open file '/content/main_gpu.py': [Errno 2] No such file or directory` cho thấy rằng file `main_gpu.py` chưa tồn tại.

In [ ]:
!python main_gpu.py --dataset ACM --model FastGTN --num_layers 3 --epoch 200 --lr 0.05 --channel_agg mean --num_channels 2 --non_local_weight -1 --K 1 --non_local

python3: can't open file '/content/main_gpu.py': [Errno 2] No such file or directory


### 16. Lưu Dữ liệu đã Xử lý

Bước này lưu các đối tượng Python đã được xử lý (`node_faeture`, `edges`, `labels`) vào các file pickle trong thư mục `data/ACM/`. Điều này giúp chuẩn bị dữ liệu ở định dạng mà script huấn luyện mô hình có thể dễ dàng tải và sử dụng.

-   `node_features.pkl`: Chứa ma trận đặc trưng nút.
-   `edges.pkl`: Chứa danh sách các ma trận kề.
-   `labels.pkl`: Chứa các nhãn (train, validation, test).

In [ ]:
import os
import pickle

# Create the directory structure if it doesn't exist
output_dir = 'data/ACM'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save node_faeture as node_features.pkl
with open(os.path.join(output_dir, 'node_features.pkl'), 'wb') as f:
    pickle.dump(node_faeture, f) # node_faeture is a numpy array

# Save edges as edges.pkl
# The 'edges' variable in the kernel state is already a list of sparse matrices (csr/csc_matrix),
# which is the format expected by the model script.
with open(os.path.join(output_dir, 'edges.pkl'), 'wb') as f:
    pickle.dump(edges, f)

# Save labels as labels.pkl
# The 'labels' variable in the kernel state is a list of numpy arrays for train, valid, test labels.
with open(os.path.join(output_dir, 'labels.pkl'), 'wb') as f:
    pickle.dump(labels, f)

print(f"Saved node_features.pkl, edges.pkl, and labels.pkl to {os.path.abspath(output_dir)}/")

Saved node_features.pkl, edges.pkl, and labels.pkl to /content/data/ACM/


### 17. Script huấn luyện mô hình (`main_gpu.py`)

Cell này chứa nội dung của script `main_gpu.py` và thực thi nó. Script này là chương trình chính để huấn luyện các mô hình GNN (GTN hoặc FastGTN) trên dataset ACM. Nó bao gồm:

-   **Khởi tạo hạt giống (seed)**: Để đảm bảo khả năng tái tạo kết quả.
-   **Phân tích đối số**: Sử dụng `argparse` để xử lý các tham số cấu hình mô hình như dataset, số epoch, learning rate, v.v.
-   **Tải dữ liệu**: Đọc `node_features.pkl`, `edges.pkl`, `labels.pkl` đã lưu trước đó.
-   **Xây dựng ma trận kề**: Chuyển đổi các ma trận kề từ định dạng `scipy.sparse` sang `torch.Tensor` và chuẩn hóa chúng.
-   **Phân chia dữ liệu**: Chuẩn bị dữ liệu huấn luyện, validation và kiểm tra cùng với nhãn.
-   **Khởi tạo mô hình**: Tạo instance của mô hình `GTN` hoặc `FastGTN` dựa trên đối số `args.model`.
-   **Huấn luyện và Đánh giá**: Lặp qua số epoch đã định, thực hiện forward pass, tính toán loss, backward pass, và cập nhật trọng số. Đánh giá mô hình trên tập validation và test, lưu lại kết quả F1-score tốt nhất.
-   **Lặp lại các lần chạy**: Thực hiện nhiều lần chạy (`runs`) để tính toán độ ổn định và độ lệch chuẩn của kết quả.

Phần `if '__file__' not in locals():` được thêm vào để mô phỏng việc truyền đối số dòng lệnh khi chạy trong môi trường Colab tương tác, giúp script có thể nhận các tham số như khi chạy từ terminal.

In [ ]:
import torch
import numpy as np
import torch.nn as nn
from model_gtn import GTN
from model_fastgtn import FastGTNs
import pickle
import argparse
from torch_geometric.utils import add_self_loops
from sklearn.metrics import f1_score as sk_f1_score
from utils import init_seed, _norm
import copy
import os
import sys

# Content of inits.py, which was missing and caused a previous ModuleNotFoundError
inits_content = """
import torch
import math
from torch.nn import init

def glorot(tensor):
    if tensor is not None:
        stdv = math.sqrt(2.0 / (tensor.size(-2) + tensor.size(-1)))
        init.normal_(tensor, 0.0, stdv)

def zeros(tensor):
    if tensor is not None:
        init.zeros_(tensor)
"""

# Create the inits.py file if it doesn't exist
inits_path = 'inits.py'
if not os.path.exists(inits_path):
    with open(inits_path, 'w') as f:
        f.write(inits_content)
    print(f"Created {inits_path} to resolve ModuleNotFoundError.")

if __name__ == '__main__':
    init_seed(seed=777)
    parser = argparse.ArgumentParser()
    parser.add_argument('--model', type=str, default='GTN',
                        help='Model')
    parser.add_argument('--dataset', type=str,
                        help='Dataset')
    parser.add_argument('--epoch', type=int, default=200,
                        help='Training Epochs')
    parser.add_argument('--node_dim', type=int, default=64,
                        help='hidden dimensions')
    parser.add_argument('--num_channels', type=int, default=2,
                        help='number of channels')
    parser.add_argument('--lr', type=float, default=0.01,
                        help='learning rate')
    parser.add_argument('--weight_decay', type=float, default=0.001,
                        help='l2 reg')
    parser.add_argument('--num_layers', type=int, default=1,
                        help='number of GT/FastGT layers')
    parser.add_argument('--runs', type=int, default=10,
                        help='number of runs')
    parser.add_argument("--channel_agg", type=str, default='concat')
    parser.add_argument("--remove_self_loops", action='store_true', help="remove_self_loops")
    # Configurations for FastGTNs
    parser.add_argument("--non_local", action='store_true', help="use non local operations")
    parser.add_argument("--non_local_weight", type=float, default=0, help="weight initialization for non local operations")
    parser.add_argument("--beta", type=float, default=0, help="beta (Identity matrix)")
    parser.add_argument('--K', type=int, default=1,
                        help='number of non-local negibors')
    parser.add_argument("--pre_train", action='store_true', help="pre-training FastGT layers")
    parser.add_argument('--num_FastGTN_layers', type=int, default=1,
                        help='number of FastGTN layers')

    # This block handles Colab-specific arguments that interfere with argparse
    # sys.argv will contain the script name and other kernel arguments like -f
    # We extract only the relevant arguments for our script.
    if '__file__' not in locals(): # Check if running in an interactive environment like Colab
        # Save original argv and then construct desired args
        original_argv = sys.argv
        sys.argv = [original_argv[0]]
        # Add the arguments intended for the script from the previous execution attempts
        sys.argv.extend([
            '--dataset', 'ACM',
            '--model', 'FastGTN',
            '--num_layers', '3',
            '--epoch', '200',
            '--lr', '0.05',
            '--channel_agg', 'mean',
            '--num_channels', '2'
        ])
        print("Simulating command-line arguments for argparse:", sys.argv[1:])

    args = parser.parse_args()
    print(args)

    epochs = args.epoch
    node_dim = args.node_dim
    num_channels = args.num_channels
    lr = args.lr
    weight_decay = args.weight_decay
    num_layers = args.num_layers

    # Corrected path: load from './data/ACM/' instead of '../data/ACM/'
    with open('data/%s/node_features.pkl' % args.dataset,'rb') as f:
        node_features = pickle.load(f)
    with open('data/%s/edges.pkl' % args.dataset,'rb') as f:
        edges = pickle.load(f)
    with open('data/%s/labels.pkl' % args.dataset,'rb') as f:
        labels = pickle.load(f)
    if args.dataset == 'PPI':
        with open('data/%s/ppi_tvt_nids.pkl' % args.dataset, 'rb') as fp:
            nids = pickle.load(fp)

    num_nodes = edges[0].shape[0]
    args.num_nodes = num_nodes
    # build adjacency matrices for each edge type
    A = []
    for i,edge in enumerate(edges):
        edge_tmp = torch.from_numpy(np.vstack((edge.nonzero()[1], edge.nonzero()[0]))).type(torch.cuda.LongTensor)
        value_tmp = torch.ones(edge_tmp.shape[1]).type(torch.cuda.FloatTensor)
        # normalize each adjacency matrix
        if args.model == 'FastGTN' and args.dataset != 'AIRPORT':
            edge_tmp, value_tmp = add_self_loops(edge_tmp, edge_attr=value_tmp, fill_value=1e-20, num_nodes=num_nodes)
            deg_inv_sqrt, deg_row, deg_col = _norm(edge_tmp.detach(), num_nodes, value_tmp.detach())
            value_tmp = deg_inv_sqrt[deg_row] * value_tmp
        A.append((edge_tmp,value_tmp))
    edge_tmp = torch.stack((torch.arange(0,num_nodes),torch.arange(0,num_nodes))).type(torch.cuda.LongTensor)
    value_tmp = torch.ones(num_nodes).type(torch.cuda.FloatTensor)
    A.append((edge_tmp,value_tmp))


    num_edge_type = len(A)
    node_features = torch.from_numpy(node_features).type(torch.cuda.FloatTensor)
    if args.dataset == 'PPI':
        train_node = torch.from_numpy(nids[0]).type(torch.cuda.LongTensor)
        train_target = torch.from_numpy(labels[nids[0]]).type(torch.cuda.FloatTensor)
        valid_node = torch.from_numpy(nids[1]).type(torch.cuda.LongTensor)
        valid_target = torch.from_numpy(labels[nids[1]]).type(torch.cuda.FloatTensor)
        test_node = torch.from_numpy(nids[2]).type(torch.cuda.LongTensor)
        test_target = torch.from_numpy(labels[nids[2]]).type(torch.cuda.FloatTensor)
        num_classes = 121
        is_ppi = True
    else:
        train_node = torch.from_numpy(np.array(labels[0])[:,0]).type(torch.cuda.LongTensor)
        train_target = torch.from_numpy(np.array(labels[0])[:,1]).type(torch.cuda.LongTensor)
        valid_node = torch.from_numpy(np.array(labels[1])[:,0]).type(torch.cuda.LongTensor)
        valid_target = torch.from_numpy(np.array(labels[1])[:,1]).type(torch.cuda.LongTensor)
        test_node = torch.from_numpy(np.array(labels[2])[:,0]).type(torch.cuda.LongTensor)
        test_target = torch.from_numpy(np.array(labels[2])[:,1]).type(torch.cuda.LongTensor)
        num_classes = np.max([torch.max(train_target).item(), torch.max(valid_target).item(), torch.max(test_target).item()])+1
        is_ppi = False
    final_f1, final_micro_f1 = [], []
    tmp = None
    runs = args.runs
    if args.pre_train:
        runs += 1
        pre_trained_fastGTNs = None
    for l in range(runs):
        # initialize a model
        if args.model == 'GTN':
            model = GTN(num_edge=len(A),
                                num_channels=num_channels,
                                w_in = node_features.shape[1],
                                w_out = node_dim,
                                num_class=num_classes,
                                num_layers=num_layers,
                                num_nodes=num_nodes,
                                args=args)
        elif args.model == 'FastGTN':
            if args.pre_train and l == 1:
                pre_trained_fastGTNs = []
                for layer in range(args.num_FastGTN_layers):
                    pre_trained_fastGTNs.append(copy.deepcopy(model.fastGTNs[layer].layers))
            while len(A) > num_edge_type:
                del A[-1]
            model = FastGTNs(num_edge_type=len(A),
                            w_in = node_features.shape[1],
                            num_class=num_classes,
                            num_nodes = node_features.shape[0],
                            args = args)
            if args.pre_train and l > 0:
                for layer in range(args.num_FastGTN_layers):
                    model.fastGTNs[layer].layers = pre_trained_fastGTNs[layer]

        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        model.cuda()
        if args.dataset == 'PPI':
            loss = nn.BCELoss()
        else:
            loss = nn.CrossEntropyLoss()
        Ws = []

        best_val_loss = 10000
        best_test_loss = 10000
        best_train_loss = 10000
        best_train_f1, best_micro_train_f1 = 0, 0
        best_val_f1, best_micro_val_f1 = 0, 0
        best_test_f1, best_micro_test_f1 = 0, 0

        for i in range(epochs):
            # print('Epoch ',i)
            model.zero_grad()
            model.train()
            if args.model == 'FastGTN':
                loss,y_train,W = model(A, node_features, train_node, train_target, epoch=i)
            else:
                loss,y_train,W = model(A, node_features, train_node, train_target)
            if args.dataset == 'PPI':
                y_train = (y_train > 0).detach().float().cpu()
                train_f1 = 0.0
                sk_train_f1 = sk_f1_score(train_target.detach().cpu().numpy(), y_train.numpy(), average='micro')
            else:
                train_f1 = sk_f1_score(train_target.detach().cpu(), torch.argmax(y_train.detach(),dim=1).cpu(), average='macro')
                sk_train_f1 = sk_f1_score(train_target.detach().cpu(), np.argmax(y_train.detach().cpu(), axis=1), average='micro')
            # print(W)
            # print('Train - Loss: {}, Macro_F1: {}, Micro_F1: {}'.format(loss.detach().cpu().numpy(), train_f1, sk_train_f1))

            loss.backward()
            optimizer.step()
            model.eval()
            # Valid
            with torch.no_grad():
                if args.model == 'FastGTN':
                    val_loss, y_valid,_ = model.forward(A, node_features, valid_node, valid_target, epoch=i)
                else:
                    val_loss, y_valid,_ = model.forward(A, node_features, valid_node, valid_target)
                if args.dataset == 'PPI':
                    val_f1 = 0.0
                    y_valid = (y_valid > 0).detach().float().cpu()
                    sk_val_f1 = sk_f1_score(valid_target.detach().cpu().numpy(), y_valid.numpy(), average='micro')
                else:
                    val_f1 = sk_f1_score(valid_target.detach().cpu(), torch.argmax(y_valid.detach(),dim=1).cpu(), average='macro')
                    sk_val_f1 = sk_f1_score(valid_target.detach().cpu(), np.argmax(y_valid.detach().cpu(), axis=1), average='micro')
                # print('Valid - Loss: {}, Macro_F1: {}, Micro_F1: {}'.format(val_loss.detach().cpu().numpy(), val_f1, sk_val_f1))

                if args.model == 'FastGTN':
                    test_loss, y_test,W = model.forward(A, node_features, test_node, test_target, epoch=i)
                else:
                    test_loss, y_test,W = model.forward(A, node_features, test_node, test_target)
                if args.dataset == 'PPI':
                    test_f1 = 0.0
                    y_test = (y_test > 0).detach().float().cpu()
                    sk_test_f1 = sk_f1_score(test_target.detach().cpu().numpy(), y_test.numpy(), average='micro')
                else:
                    test_f1 = sk_f1_score(test_target.detach().cpu(), torch.argmax(y_test.detach(),dim=1).cpu(), average='macro')
                    sk_test_f1 = sk_f1_score(test_target.detach().cpu(), np.argmax(y_test.detach().cpu(), axis=1), average='micro')
                # print('Test - Loss: {}, Macro_F1: {}, Micro_F1:{} \n'.format(test_loss.detach().cpu().numpy(), test_f1, sk_test_f1))
            if sk_val_f1 > best_micro_val_f1:
                best_val_loss = val_loss.detach().cpu().numpy()
                best_test_loss = test_loss.detach().cpu().numpy()
                best_train_loss = loss.detach().cpu().numpy()
                best_train_f1 = train_f1
                best_val_f1 = val_f1
                best_test_f1 = test_f1
                best_micro_train_f1 = sk_train_f1
                best_micro_val_f1 = sk_val_f1
                best_micro_test_f1 = sk_test_f1
        if l == 0 and args.pre_train:
            continue
        print('Run {}'.format(l))
        print('--------------------Best Result-------------------------')
        print('Train - Loss: {:.4f}, Macro_F1: {:.4f}, Micro_F1: {:.4f}'.format(best_test_loss, best_train_f1, best_micro_train_f1))
        print('Valid - Loss: {:.4f}, Macro_F1: {:.4f}, Micro_F1: {:.4f}'.format(best_val_loss, best_val_f1, best_micro_val_f1))
        print('Test - Loss: {:.4f}, Macro_F1: {:.4f}, Micro_F1: {:.4f}'.format(best_test_loss, best_test_f1, best_micro_test_f1))
        final_f1.append(best_test_f1)
        final_micro_f1.append(best_micro_test_f1)

    print('--------------------Final Result-------------------------')
    print('Test - Macro_F1: {:.4f}+{:.4f}, Micro_F1:{:.4f}+{:.4f}'.format(np.mean(final_f1), np.std(final_f1), np.mean(final_micro_f1), np.std(final_micro_f1)))

Simulating command-line arguments for argparse: ['--dataset', 'ACM', '--model', 'FastGTN', '--num_layers', '3', '--epoch', '200', '--lr', '0.05', '--channel_agg', 'mean', '--num_channels', '2']
Namespace(model='FastGTN', dataset='ACM', epoch=200, node_dim=64, num_channels=2, lr=0.05, weight_decay=0.001, num_layers=3, runs=10, channel_agg='mean', remove_self_loops=False, non_local=False, non_local_weight=0, beta=0, K=1, pre_train=False, num_FastGTN_layers=1)
Run 0
--------------------Best Result-------------------------
Train - Loss: 0.8666, Macro_F1: 0.8678, Micro_F1: 0.8683
Valid - Loss: 0.7958, Macro_F1: 0.6577, Micro_F1: 0.6700
Test - Loss: 0.8666, Macro_F1: 0.6039, Micro_F1: 0.6216
Run 1
--------------------Best Result-------------------------
Train - Loss: 0.8900, Macro_F1: 0.8534, Micro_F1: 0.8567
Valid - Loss: 0.8161, Macro_F1: 0.6360, Micro_F1: 0.6500
Test - Loss: 0.8900, Macro_F1: 0.5980, Micro_F1: 0.6207
Run 2
--------------------Best Result-------------------------
Train - L

### 18. Định nghĩa Mô hình GTN (`model_gtn.py`)

Cell này tạo file `model_gtn.py`, chứa định nghĩa của mô hình Graph Transformer Networks (GTN). Mô hình GTN học cách tạo ra các meta-path (đường đi phức tạp giữa các nút) bằng cách học các ma trận trọng số cho các loại cạnh khác nhau. Các thành phần chính bao gồm:

-   `GTNLayer`: Lớp cốt lõi để tạo ra các ma trận kề tổng hợp.
-   Các lớp tuyến tính (`nn.Linear`) và dropout (`nn.Dropout`) để xử lý đặc trưng nút và phân loại.
-   Hàm mất mát (`nn.CrossEntropyLoss` hoặc `nn.BCEWithLogitsLoss`) để huấn luyện.
-   Cơ chế kết hợp các kênh (channel aggregation) như `mean`, `max`, hoặc `concat` để tổng hợp thông tin từ nhiều meta-path.

In [ ]:
import os

# Content for model_gtn.py
model_gtn_content = """
import torch
import torch.nn as nn
import torch.nn.functional as F
from layer import GTNLayer, GTNFixed
from inits import glorot, zeros

class GTN(nn.Module):
    def __init__(self, num_edge, num_channels, w_in, w_out, num_class, num_layers, num_nodes, args):
        super(GTN, self).__init__()
        self.num_edge = num_edge
        self.num_channels = num_channels
        self.w_in = w_in
        self.w_out = w_out
        self.num_class = num_class
        self.num_layers = num_layers
        self.num_nodes = num_nodes
        self.args = args

        self.is_ppi = (self.args.dataset == 'PPI')
        self.is_multilabel = self.is_ppi

        self.GTN_layers = nn.ModuleList()
        for i in range(self.num_layers):
            if i == 0:
                self.GTN_layers.append(GTNLayer(num_edge, num_channels, num_nodes, first=True))
            else:
                self.GTN_layers.append(GTNLayer(num_edge, num_channels, num_nodes, first=False))

        self.weight = nn.Parameter(torch.Tensor(self.w_in, self.w_out))
        self.bias = nn.Parameter(torch.Tensor(self.w_out))

        self.loss = nn.CrossEntropyLoss()
        self.attention = nn.Linear(self.w_out*self.num_channels, self.w_out)

        self.linear1 = nn.Linear(self.w_out, self.w_out)
        self.linear2 = nn.Linear(self.w_out, self.num_class)
        self.dropout = nn.Dropout(0.5)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight)
        zeros(self.bias)

    def forward(self, A, X, train_node, train_target):
        Ws = []
        for i in range(self.num_layers):
            if i==0:
                H, W = self.GTN_layers[i](A)
            else:
                H, W = self.GTN_layers[i](A, H)
            Ws.append(W)

        # use last layer's Ws
        # H_flatten = H.view(self.num_nodes, -1)
        # H_flatten = torch.matmul(H_flatten, self.attention.weight.transpose(1,0)) + self.attention.bias

        # combine all Ws into one adjacency matrix
        # print(Ws[-1].shape)
        # combine H from multiple channels

        # H_mean = torch.mean(H,dim=1)
        if self.args.channel_agg == 'mean':
            H_all = torch.mean(H, dim=1)
        elif self.args.channel_agg == 'max':
            H_all = torch.max(H, dim=1)[0]
        elif self.args.channel_agg == 'concat':
            H_all = H.view(self.num_nodes, -1)
            H_all = self.attention(H_all)
        else:
            raise NotImplementedError('Unknown channel aggregation method')

        X_ = torch.matmul(X, self.weight) + self.bias
        X_ = F.relu(X_)
        X_ = self.dropout(X_)

        X_ = self.linear1(X_)
        X_ = F.relu(X_)
        X_ = self.dropout(X_)
        X_ = self.linear2(X_)

        if not self.is_multilabel:
            X_ = F.log_softmax(X_, dim=1)

        if self.training:
            if self.is_multilabel:
                loss = self.loss(X_[train_node], train_target)
            else:
                loss = self.loss(X_[train_node], train_target)
            return loss, X_[train_node], Ws
        else:
            if self.is_multilabel:
                return X_[train_node], Ws
            else:
                return X_[train_node], Ws

class GTNFixed(nn.Module):
    def __init__(self, num_edge, num_channels, w_in, w_out, num_class, num_layers, num_nodes, args):
        super(GTNFixed, self).__init__()
        self.num_edge = num_edge
        self.num_channels = num_channels
        self.w_in = w_in
        self.w_out = w_out
        self.num_class = num_class
        self.num_layers = num_layers
        self.num_nodes = num_nodes
        self.args = args

        self.is_ppi = (self.args.dataset == 'PPI')
        self.is_multilabel = self.is_ppi

        self.GTN_layers = nn.ModuleList()
        for i in range(self.num_layers):
            if i == 0:
                self.GTN_layers.append(GTNLayer(num_edge, num_channels, num_nodes, first=True))
            else:
                self.GTN_layers.append(GTNLayer(num_edge, num_channels, num_nodes, first=False))

        self.weight = nn.Parameter(torch.Tensor(self.w_in, self.w_out))
        self.bias = nn.Parameter(torch.Tensor(self.w_out))

        self.loss = nn.CrossEntropyLoss()
        self.attention = nn.Linear(self.w_out * self.num_channels, self.w_out)

        self.linear1 = nn.Linear(self.w_out, self.w_out)
        self.linear2 = nn.Linear(self.w_out, self.num_class)
        self.dropout = nn.Dropout(0.5)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight)
        zeros(self.bias)

    def forward(self, A, X, train_node, train_target):
        Ws = []
        for i in range(self.num_layers):
            if i==0:
                H, W = self.GTN_layers[i](A)
            else:
                H, W = self.GTN_layers[i](A, H)
            Ws.append(W)

        # use last layer's Ws
        # H_flatten = H.view(self.num_nodes, -1)
        # H_flatten = torch.matmul(H_flatten, self.attention.weight.transpose(1,0)) + self.attention.bias

        # combine all Ws into one adjacency matrix
        # print(Ws[-1].shape)
        # combine H from multiple channels

        # H_mean = torch.mean(H,dim=1)
        if self.args.channel_agg == 'mean':
            H_all = torch.mean(H, dim=1)
        elif self.args.channel_agg == 'max':
            H_all = torch.max(H, dim=1)[0]
        elif self.args.channel_agg == 'concat':
            H_all = H.view(self.num_nodes, -1)
            H_all = self.attention(H_all)
        else:
            raise NotImplementedError('Unknown channel aggregation method')

        X_ = torch.matmul(X, self.weight) + self.bias
        X_ = F.relu(X_)
        X_ = self.dropout(X_)

        X_ = self.linear1(X_)
        X_ = F.relu(X_)
        X_ = self.dropout(X_)
        X_ = self.linear2(X_)

        if not self.is_multilabel:
            X_ = F.log_softmax(X_, dim=1)

        if self.training:
            if self.is_multilabel:
                loss = self.loss(X_[train_node], train_target)
            else:
                loss = self.loss(X_[train_node], train_target)
            return loss, X_[train_node], Ws
        else:
            if self.is_multilabel:
                return X_[train_node], Ws
            else:
                return X_[train_node], Ws
"""

# Create the model_gtn.py file
model_gtn_path = 'model_gtn.py'
if not os.path.exists(model_gtn_path):
    with open(model_gtn_path, 'w') as f:
        f.write(model_gtn_content)
    print(f"Created {model_gtn_path}.")

### 19. Định nghĩa Lớp `GTNLayer` (`layer.py`)

Cell này tạo file `layer.py`, chứa định nghĩa của lớp `GTNLayer` được sử dụng trong mô hình GTN. Lớp này nhận một tập hợp các ma trận kề đầu vào (là `A`, đại diện cho các loại cạnh khác nhau trong đồ thị dị thể) và sử dụng các trọng số học được (`self.weight`) để linh hoạt kết hợp chúng, tạo ra các ma trận kề meta-path cho từng kênh (channel).

-   `self.weight`: Tham số trọng số học được, có kích thước `(num_channels, num_edge, 1, 1)`. Các trọng số này được sử dụng để gán mức độ quan trọng khác nhau cho mỗi loại cạnh trong mỗi kênh, từ đó định hình các meta-path.
-   `forward(self, A, H_=None)`: Phương thức tính toán chính:
    1.  Nó nhân từng ma trận kề ban đầu trong `A` (đã được làm nổi lên một chiều) với `self.weight` (đã được điều chỉnh kích thước) và sau đó tổng hợp chúng theo chiều `num_edge` để tạo ra các ma trận kề meta-path (`num_channels` ma trận, mỗi ma trận có kích thước `num_nodes x num_nodes`).
    2.  Áp dụng hàm softmax trên các ma trận kề meta-path này để chuẩn hóa chúng.
    3.  Nếu đây là lớp `GTNLayer` đầu tiên (`first=True`), nó tính toán ma trận kề tổng hợp `H` bằng cách nhân các ma trận kề meta-path đã học với chính nó (`torch.matmul(A,A)`).
    4.  Nếu không phải là lớp đầu tiên (`first=False`), nó tính toán `H` bằng cách nhân các ma trận kề meta-path đã học với ma trận `H_` từ lớp `GTNLayer` trước đó.
    5.  Hàm `forward` trả về ma trận kề tổng hợp cuối cùng `H` và tập hợp các ma trận kề meta-path đã học (đã qua softmax) `A`.

In [ ]:
import os

# Content for layer.py (dependency for model_gtn.py)
layer_content = """
import torch
import torch.nn as nn
import torch.nn.functional as F
from inits import glorot, zeros

class GTNLayer(nn.Module):

    def __init__(self, num_edge, num_channels, num_nodes, first=True):
        super(GTNLayer, self).__init__()
        self.num_edge = num_edge
        self.num_channels = num_channels
        self.num_nodes = num_nodes
        self.first = first
        self.weight = nn.Parameter(torch.Tensor(num_channels, num_edge,1,1))
        self.bias = None
        self.reset_parameters()

    def reset_parameters(self):
        glorot(self.weight)

    def forward(self, A, H_=None):
        if self.first:
            a = self.weight.view(self.num_channels, self.num_edge, 1, 1)
            A = a.cuda() * A.unsqueeze(0).cuda()
            A = A.sum(dim=1)
            A = F.softmax(A, dim=2)
            H = torch.matmul(A,A)
            return H, A
        else:
            a = self.weight.view(self.num_channels, self.num_edge, 1, 1)
            A = a.cuda() * A.unsqueeze(0).cuda()
            A = A.sum(dim=1)
            A = F.softmax(A, dim=2)
            H = torch.matmul(A, H_)
            return H, A
"""

# Create the layer.py file
layer_path = 'layer.py'
if not os.path.exists(layer_path):
    with open(layer_path, 'w') as f:
        f.write(layer_content)
    print(f"Created {layer_path}.")

Created layer.py.


### 20. Định nghĩa Mô hình FastGTN (`model_fastgtn.py`)

Cell này tạo file `model_fastgtn.py`, chứa định nghĩa của mô hình FastGTN. FastGTN là một phiên bản cải tiến của GTN, tập trung vào hiệu quả và khả năng mở rộng. Các thành phần chính tương tự như GTN nhưng có thể bao gồm:

-   `FastGTNLayer`: Lớp cốt lõi cho FastGTN.
-   `SkipConnection`: Một lớp tùy chọn để thêm kết nối bỏ qua, có thể sử dụng các phép toán non-local để nắm bắt các phụ thuộc xa.
-   Cũng sử dụng các lớp tuyến tính, dropout và hàm mất mát tương tự như GTN.

In [ ]:
import os

# Content for model_fastgtn.py
model_fastgtn_content = """
import torch
import torch.nn as nn
import torch.nn.functional as F
from layer_fastgtn import FastGTNLayer, SkipConnection
from inits import glorot, zeros


class FastGTNs(nn.Module):
    def __init__(self, num_edge_type, w_in, num_class, num_nodes, args):
        super(FastGTNs, self).__init__()
        self.num_edge_type = num_edge_type
        self.w_in = w_in
        self.num_class = num_class
        self.num_nodes = num_nodes
        self.args = args

        self.is_ppi = (self.args.dataset == 'PPI')
        self.is_multilabel = self.is_ppi

        self.fastGTNs = nn.ModuleList()
        for i in range(self.args.num_FastGTN_layers):
            if i == 0:
                self.fastGTNs.append(FastGTNLayer(num_edge_type, self.args.num_channels, self.num_nodes, first=True,
                                                args=args))
            else:
                self.fastGTNs.append(FastGTNLayer(num_edge_type, self.args.num_channels, self.num_nodes,
                                                first=False, args=args))

        if self.args.channel_agg == 'mean' or self.args.channel_agg == 'max':
            self.attention = nn.Linear(self.args.node_dim, self.args.node_dim)
        elif self.args.channel_agg == 'concat':
            self.attention = nn.Linear(self.args.node_dim * self.args.num_channels, self.args.node_dim)


        self.final_linear = nn.Linear(self.args.node_dim * self.args.num_FastGTN_layers, self.num_class)
        self

        # self.weight = nn.Parameter(torch.Tensor(self.w_in, self.args.node_dim))
        # self.bias = nn.Parameter(torch.Tensor(self.args.node_dim))
        self.dropout = nn.Dropout(self.args.dropout)


        if self.is_multilabel:
            self.loss = nn.BCEWithLogitsLoss()
        else:
            self.loss = nn.CrossEntropyLoss()
        self.reset_parameters()

    def reset_parameters(self):
        # glorot(self.weight)
        # zeros(self.bias)
        nn.init.xavier_uniform_(self.final_linear.weight)
        zeros(self.final_linear.bias)

    def forward(self, A, X, train_node, train_target, epoch):
        Xs = []
        for i in range(self.args.num_FastGTN_layers):
            if i==0:
                X_ = self.fastGTNs[i](A, X, epoch)
            else:
                X_ = self.fastGTNs[i](A, X_)
            Xs.append(X_)

        if self.args.channel_agg == 'mean':
            X_all = torch.mean(Xs[-1], dim=1)
        elif self.args.channel_agg == 'max':
            X_all = torch.max(Xs[-1], dim=1)[0]
        elif self.args.channel_agg == 'concat':
            X_all = Xs[-1].view(self.num_nodes, -1)
        else:
            raise NotImplementedError('Unknown channel aggregation method')


        # for i in range(len(Xs)):
        #     if self.args.channel_agg == 'mean':
        #         Xs[i] = torch.mean(Xs[i], dim=1)
        #     elif self.args.channel_agg == 'max':
        #         Xs[i] = torch.max(Xs[i], dim=1)[0]
        #     elif self.args.channel_agg == 'concat':
        #         Xs[i] = Xs[i].view(self.num_nodes, -1)

        #     Xs[i] = self.dropout(F.relu(self.attention(Xs[i])))

        # X_out = torch.cat(Xs, dim=1)
        # X_out = self.dropout(F.relu(self.attention(X_out)))

        X_out = self.final_linear(self.dropout(F.relu(X_all)))

        if self.is_multilabel:
            pred = X_out
        else:
            pred = F.log_softmax(X_out, dim=1)

        if self.training:
            if self.is_multilabel:
                loss = self.loss(pred[train_node], train_target)
            else:
                loss = self.loss(pred[train_node], train_target)
            return loss, pred[train_node], X_all
        else:
            if self.is_multilabel:
                return pred[train_node], X_all
            else:
                return pred[train_node], X_all
"""

# Create the model_fastgtn.py file
model_fastgtn_path = 'model_fastgtn.py'
if not os.path.exists(model_fastgtn_path):
    with open(model_fastgtn_path, 'w') as f:
        f.write(model_fastgtn_content)
    print(f"Created {model_fastgtn_path}.")

### 21. Định nghĩa Lớp FastGTN (`layer_fastgtn.py`)

Cell này tạo file `layer_fastgtn.py`, chứa định nghĩa của `FastGTNLayer` và `SkipConnection` được sử dụng trong mô hình FastGTN.

-   `FastGTNLayer`:
    -   Tương tự như `GTNLayer` nhưng được tối ưu hóa cho hiệu suất.
    -   Có thể bao gồm các lớp tuyến tính ban đầu để biến đổi đặc trưng nút.
    -   Tích hợp `SkipConnection` để xử lý phi cục bộ nếu được cấu hình.

-   `SkipConnection`:
    -   Thực hiện các phép toán phi cục bộ (`non_local_op`) trên đặc trưng nút.
    -   Sử dụng tham số `non_local_weight` để kiểm soát ảnh hưởng của kết nối phi cục bộ.

In [ ]:
import os

# Content for layer_fastgtn.py (dependency for model_fastgtn.py)
layer_fastgtn_content = """
import torch
import torch.nn as nn
import torch.nn.functional as F
from inits import glorot, zeros

class FastGTNLayer(nn.Module):

    def __init__(self, num_edge_type, num_channels, num_nodes, first=True, args=None):
        super(FastGTNLayer, self).__init__()
        self.num_edge_type = num_edge_type
        self.num_channels = num_channels
        self.num_nodes = num_nodes
        self.first = first
        self.args = args

        self.weight = nn.Parameter(torch.Tensor(num_channels, num_edge_type, 1, 1))
        self.bias = None

        self.skip = SkipConnection(self.num_nodes, num_channels, args)

        if self.first:
            self.linear1 = nn.Linear(self.args.w_in, self.args.node_dim)
            self.linear2 = nn.Linear(self.args.node_dim, self.args.node_dim)
        self.reset_parameters()

    def reset_parameters(self):
        glorot(self.weight)

    def forward(self, A, X, epoch=None):

        if self.first:
            X = F.relu(self.linear1(X))
            X = F.relu(self.linear2(X))

        a = self.weight.view(self.num_channels, self.num_edge_type, 1, 1)
        A = a * A.unsqueeze(0)
        A = A.sum(dim=1)
        A = F.softmax(A, dim=2)

        H = torch.matmul(A, X)

        if self.args.non_local:
            H = self.skip(H, epoch)

        return H

class SkipConnection(nn.Module):
    def __init__(self, num_nodes, num_channels, args=None):
        super(SkipConnection, self).__init__()
        self.num_nodes = num_nodes
        self.num_channels = num_channels
        self.args = args

        self.linear_in = nn.Linear(num_nodes, num_nodes)
        self.linear_out = nn.Linear(num_nodes, num_nodes)
        self.weight = nn.Parameter(torch.Tensor(1, num_nodes, num_nodes))
        self.non_local_weight = self.args.non_local_weight
        self.reset_parameters()

    def reset_parameters(self):
        if self.non_local_weight == 0:
            nn.init.constant_(self.weight, 0)
        else:
            glorot(self.weight)

    def forward(self, X, epoch=None):
        H_origin = X

        X_non_local = []
        for i in range(self.num_channels):
            X_non_local.append(self.non_local_op(X[:, i, :], self.weight))

        X_non_local = torch.stack(X_non_local, dim=1)

        if self.args.non_local_weight == 0:
            alpha = torch.sigmoid(self.non_local_weight)
        else:
            alpha = torch.sigmoid(self.non_local_weight + (epoch * 0.001))

        H = alpha * X_non_local + (1 - alpha) * H_origin

        return H

    def non_local_op(self, X, W):
        X_ = X.unsqueeze(0)
        X_ = self.linear_in(X_)

        V = F.softmax(W + X_.transpose(1,2)@X_, dim=-1) @ X_

        H = self.linear_out(V.squeeze(0))
        return H
"""

# Create the layer_fastgtn.py file
layer_fastgtn_path = 'layer_fastgtn.py'
if not os.path.exists(layer_fastgtn_path):
    with open(layer_fastgtn_path, 'w') as f:
        f.write(layer_fastgtn_content)
    print(f"Created {layer_fastgtn_path}.")

Created layer_fastgtn.py.


### 22. Tiện ích và Hàm hỗ trợ (`utils.py`)

Cell này tạo file `utils.py`, chứa các hàm tiện ích và hỗ trợ được sử dụng trong quá trình huấn luyện mô hình GNN. Các hàm chính bao gồm:

-   `init_seed(seed=777)`: Khởi tạo seed cho các thư viện `torch`, `numpy`, và `random` để đảm bảo khả năng tái tạo kết quả. Nó cũng tắt `cudnn.deterministic` và `cudnn.benchmark`.
-   `_norm(edge_index, num_nodes, edge_weight, improved=False, dtype=None)`: Chuẩn hóa ma trận kề (adjacency matrix). Hàm này tính toán bậc của các nút, sau đó sử dụng nó để tính toán các hệ số chuẩn hóa theo kiểu đối xứng (symmetric normalization) thường dùng trong GCN.
-   `add_self_loops(edge_index, edge_attr=None, fill_value=1, num_nodes=None)`: Thêm các cạnh tự vòng (self-loops) vào đồ thị. Các cạnh tự vòng giúp mỗi nút có thể tích hợp thông tin từ chính nó trong quá trình lan truyền thông điệp.
-   `maybe_num_nodes(edge_index, num_nodes=None)`: Một hàm trợ giúp để xác định số lượng nút nếu `num_nodes` không được cung cấp rõ ràng, dựa trên chỉ số nút lớn nhất trong `edge_index`.

In [ ]:
import os

# Content for utils.py
utils_content = """
import torch
import numpy as np
import random
from torch_geometric.utils import degree

def init_seed(seed=777):
    \'''
    Disable cudnn to maximize reproducibility
    \'''
    torch.cuda.manual_seed(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def _norm(edge_index, num_nodes, edge_weight, improved=False, dtype=None):
    if edge_weight is None:
        edge_weight = torch.ones((edge_index.size(1),), dtype=dtype,
                                 device=edge_index.device)

    fill_value = 1 if improved else 0
    edge_index, edge_weight = add_self_loops(edge_index, edge_weight, fill_value,
                                             num_nodes)

    row, col = edge_index
    deg = degree(row, num_nodes, dtype=edge_weight.dtype)
    deg_inv_sqrt = deg.pow_(-0.5)
    deg_inv_sqrt.masked_fill_(deg_inv_sqrt == float('inf'), 0)
    return deg_inv_sqrt[row] * edge_weight * deg_inv_sqrt[col], row, col

def add_self_loops(edge_index, edge_attr=None, fill_value=1, num_nodes=None):
    \"\"\"Adds a self-loop to every node in the graph given by :attr:`edge_index`.

    Args:
        edge_index (LongTensor): The edge indices.
        edge_attr (Tensor, optional): Edge weights or multi-dimensional edge
            features. (default: :obj:`None`)
        fill_value (float, optional): If :attr:`edge_attr` is :obj:`None`,
            this value is used to fill the self-loop corresponding edge
            features. If :attr:`edge_attr` exists, this value is added to
            the existing self-loop corresponding edge features. (default:
            :obj:`1`)
        num_nodes (int, optional): The number of nodes, *i.e.*
            :math:`|\\mathcal{V}|`. (default: :obj:`None`)

    :rtype:
        (:class:`LongTensor`, :class:`Tensor`)
    \"\"\"
    num_nodes = maybe_num_nodes(edge_index, num_nodes)
    row, col = edge_index

    mask = row == col
    if mask.any():
        edge_attr[mask] += fill_value
        return edge_index, edge_attr

    loop_index = torch.arange(0, num_nodes, dtype=row.dtype,
                              device=edge_index.device)
    loop_index = loop_index.unsqueeze(0).repeat(2, 1)

    edge_index = torch.cat([edge_index, loop_index], dim=1)
    if edge_attr is not None:
        fill_factor = 1 if isinstance(fill_value, (int, float)) else fill_value
        loop_attr = edge_attr.new_full((num_nodes, ) + edge_attr.size()[1:],
                                       fill_factor)
        edge_attr = torch.cat([edge_attr, loop_attr], dim=0)

    return edge_index, edge_attr

def maybe_num_nodes(edge_index, num_nodes=None):
    if num_nodes is not None:
        return num_nodes
    elif edge_index.numel() == 0:
        return 0
    else:
        return int(edge_index.max()) + 1
"""

# Create the utils.py file
utils_path = 'utils.py'
if not os.path.exists(utils_path):
    with open(utils_path, 'w') as f:
        f.write(utils_content)
    print(f"Created {utils_path}.")

In [ ]:
%%writefile main_gpu.py
import torch
import numpy as np
import torch.nn as nn
from model_gtn import GTN
from model_fastgtn import FastGTNs
import pickle
import argparse
from torch_geometric.utils import add_self_loops
from sklearn.metrics import f1_score as sk_f1_score
from utils import init_seed, _norm
import copy
import os
import sys

# Tạo file inits.py nếu chưa có
inits_content = """
import torch
import math
from torch.nn import init

def glorot(tensor):
    if tensor is not None:
        stdv = math.sqrt(2.0 / (tensor.size(-2) + tensor.size(-1)))
        init.normal_(tensor, 0.0, stdv)

def zeros(tensor):
    if tensor is not None:
        init.zeros_(tensor)
"""
inits_path = 'inits.py'
if not os.path.exists(inits_path):
    with open(inits_path, 'w') as f:
        f.write(inits_content)

if __name__ == '__main__':
    init_seed(seed=777)
    parser = argparse.ArgumentParser()
    parser.add_argument('--model', type=str, default='GTN', help='Model')
    parser.add_argument('--dataset', type=str, help='Dataset')
    parser.add_argument('--epoch', type=int, default=200, help='Training Epochs')
    parser.add_argument('--node_dim', type=int, default=64, help='hidden dimensions')
    parser.add_argument('--num_channels', type=int, default=2, help='number of channels')
    parser.add_argument('--lr', type=float, default=0.01, help='learning rate')
    parser.add_argument('--weight_decay', type=float, default=0.001, help='l2 reg')
    parser.add_argument('--num_layers', type=int, default=1, help='number of GT/FastGT layers')
    parser.add_argument('--runs', type=int, default=10, help='number of runs')
    parser.add_argument("--channel_agg", type=str, default='concat')
    parser.add_argument("--remove_self_loops", action='store_true', help="remove_self_loops")
    parser.add_argument("--non_local", action='store_true', help="use non local operations")
    parser.add_argument("--non_local_weight", type=float, default=0, help="weight initialization for non local operations")
    parser.add_argument("--beta", type=float, default=0, help="beta (Identity matrix)")
    parser.add_argument('--K', type=int, default=1, help='number of non-local negibors')
    parser.add_argument("--pre_train", action='store_true', help="pre-training FastGT layers")
    parser.add_argument('--num_FastGTN_layers', type=int, default=1, help='number of FastGTN layers')

    args = parser.parse_args()
    print(args)

    epochs = args.epoch
    node_dim = args.node_dim
    num_channels = args.num_channels
    lr = args.lr
    weight_decay = args.weight_decay
    num_layers = args.num_layers

    # Đọc dữ liệu đã tiền xử lý
    with open('data/%s/node_features.pkl' % args.dataset,'rb') as f:
        node_features = pickle.load(f)
    with open('data/%s/edges.pkl' % args.dataset,'rb') as f:
        edges = pickle.load(f)
    with open('data/%s/labels.pkl' % args.dataset,'rb') as f:
        labels = pickle.load(f)

    if args.dataset == 'PPI':
        with open('data/%s/ppi_tvt_nids.pkl' % args.dataset, 'rb') as fp:
            nids = pickle.load(fp)

    num_nodes = edges[0].shape[0]
    args.num_nodes = num_nodes

    # build adjacency matrices
    A = []
    for i,edge in enumerate(edges):
        edge_tmp = torch.from_numpy(np.vstack((edge.nonzero()[1], edge.nonzero()[0]))).type(torch.cuda.LongTensor)
        value_tmp = torch.ones(edge_tmp.shape[1]).type(torch.cuda.FloatTensor)
        if args.model == 'FastGTN' and args.dataset != 'AIRPORT':
            edge_tmp, value_tmp = add_self_loops(edge_tmp, edge_attr=value_tmp, fill_value=1e-20, num_nodes=num_nodes)
            deg_inv_sqrt, deg_row, deg_col = _norm(edge_tmp.detach(), num_nodes, value_tmp.detach())
            value_tmp = deg_inv_sqrt[deg_row] * value_tmp
        A.append((edge_tmp,value_tmp))

    edge_tmp = torch.stack((torch.arange(0,num_nodes),torch.arange(0,num_nodes))).type(torch.cuda.LongTensor)
    value_tmp = torch.ones(num_nodes).type(torch.cuda.FloatTensor)
    A.append((edge_tmp,value_tmp))

    num_edge_type = len(A)
    node_features = torch.from_numpy(node_features).type(torch.cuda.FloatTensor)

    if args.dataset == 'PPI':
        train_node = torch.from_numpy(nids[0]).type(torch.cuda.LongTensor)
        train_target = torch.from_numpy(labels[nids[0]]).type(torch.cuda.FloatTensor)
        valid_node = torch.from_numpy(nids[1]).type(torch.cuda.LongTensor)
        valid_target = torch.from_numpy(labels[nids[1]]).type(torch.cuda.FloatTensor)
        test_node = torch.from_numpy(nids[2]).type(torch.cuda.LongTensor)
        test_target = torch.from_numpy(labels[nids[2]]).type(torch.cuda.FloatTensor)
        num_classes = 121
        is_ppi = True
    else:
        train_node = torch.from_numpy(np.array(labels[0])[:,0]).type(torch.cuda.LongTensor)
        train_target = torch.from_numpy(np.array(labels[0])[:,1]).type(torch.cuda.LongTensor)
        valid_node = torch.from_numpy(np.array(labels[1])[:,0]).type(torch.cuda.LongTensor)
        valid_target = torch.from_numpy(np.array(labels[1])[:,1]).type(torch.cuda.LongTensor)
        test_node = torch.from_numpy(np.array(labels[2])[:,0]).type(torch.cuda.LongTensor)
        test_target = torch.from_numpy(np.array(labels[2])[:,1]).type(torch.cuda.LongTensor)
        num_classes = np.max([torch.max(train_target).item(), torch.max(valid_target).item(), torch.max(test_target).item()])+1
        is_ppi = False

    final_f1, final_micro_f1 = [], []
    runs = args.runs
    if args.pre_train:
        runs += 1
        pre_trained_fastGTNs = None

    for l in range(runs):
        if args.model == 'GTN':
            model = GTN(num_edge=len(A), num_channels=num_channels, w_in=node_features.shape[1], w_out=node_dim, num_class=num_classes, num_layers=num_layers, num_nodes=num_nodes, args=args)
        elif args.model == 'FastGTN':
            if args.pre_train and l == 1:
                pre_trained_fastGTNs = []
                for layer in range(args.num_FastGTN_layers):
                    pre_trained_fastGTNs.append(copy.deepcopy(model.fastGTNs[layer].layers))
            while len(A) > num_edge_type:
                del A[-1]
            model = FastGTNs(num_edge_type=len(A), w_in=node_features.shape[1], num_class=num_classes, num_nodes=node_features.shape[0], args=args)
            if args.pre_train and l > 0:
                for layer in range(args.num_FastGTN_layers):
                    model.fastGTNs[layer].layers = pre_trained_fastGTNs[layer]

        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        model.cuda()

        if args.dataset == 'PPI':
            loss_fn = nn.BCELoss()
        else:
            loss_fn = nn.CrossEntropyLoss()

        best_val_loss = 10000
        best_test_loss = 10000
        best_train_f1, best_micro_train_f1 = 0, 0
        best_val_f1, best_micro_val_f1 = 0, 0
        best_test_f1, best_micro_test_f1 = 0, 0

        for i in range(epochs):
            model.zero_grad()
            model.train()
            if args.model == 'FastGTN':
                loss, y_train, W = model(A, node_features, train_node, train_target, epoch=i)
            else:
                loss, y_train, W = model(A, node_features, train_node, train_target)

            if args.dataset == 'PPI':
                y_train = (y_train > 0).detach().float().cpu()
                train_f1 = 0.0
                sk_train_f1 = sk_f1_score(train_target.detach().cpu().numpy(), y_train.numpy(), average='micro')
            else:
                train_f1 = sk_f1_score(train_target.detach().cpu(), torch.argmax(y_train.detach(),dim=1).cpu(), average='macro')
                sk_train_f1 = sk_f1_score(train_target.detach().cpu(), np.argmax(y_train.detach().cpu(), axis=1), average='micro')

            loss.backward()
            optimizer.step()
            model.eval()

            with torch.no_grad():
                if args.model == 'FastGTN':
                    val_loss, y_valid, _ = model.forward(A, node_features, valid_node, valid_target, epoch=i)
                else:
                    val_loss, y_valid, _ = model.forward(A, node_features, valid_node, valid_target)

                if args.dataset == 'PPI':
                    val_f1 = 0.0
                    y_valid = (y_valid > 0).detach().float().cpu()
                    sk_val_f1 = sk_f1_score(valid_target.detach().cpu().numpy(), y_valid.numpy(), average='micro')
                else:
                    val_f1 = sk_f1_score(valid_target.detach().cpu(), torch.argmax(y_valid.detach(),dim=1).cpu(), average='macro')
                    sk_val_f1 = sk_f1_score(valid_target.detach().cpu(), np.argmax(y_valid.detach().cpu(), axis=1), average='micro')

                if args.model == 'FastGTN':
                    test_loss, y_test, W = model.forward(A, node_features, test_node, test_target, epoch=i)
                else:
                    test_loss, y_test, W = model.forward(A, node_features, test_node, test_target)

                if args.dataset == 'PPI':
                    test_f1 = 0.0
                    y_test = (y_test > 0).detach().float().cpu()
                    sk_test_f1 = sk_f1_score(test_target.detach().cpu().numpy(), y_test.numpy(), average='micro')
                else:
                    test_f1 = sk_f1_score(test_target.detach().cpu(), torch.argmax(y_test.detach(),dim=1).cpu(), average='macro')
                    sk_test_f1 = sk_f1_score(test_target.detach().cpu(), np.argmax(y_test.detach().cpu(), axis=1), average='micro')

            if sk_val_f1 > best_micro_val_f1:
                best_val_loss = val_loss.detach().cpu().numpy()
                best_test_loss = test_loss.detach().cpu().numpy()
                best_train_f1 = train_f1
                best_val_f1 = val_f1
                best_test_f1 = test_f1
                best_micro_train_f1 = sk_train_f1
                best_micro_val_f1 = sk_val_f1
                best_micro_test_f1 = sk_test_f1

        if l == 0 and args.pre_train:
            continue
        print('Run {}'.format(l))
        print('--------------------Best Result-------------------------')
        print('Train - Loss: {:.4f}, Macro_F1: {:.4f}, Micro_F1: {:.4f}'.format(best_test_loss, best_train_f1, best_micro_train_f1))
        print('Valid - Loss: {:.4f}, Macro_F1: {:.4f}, Micro_F1: {:.4f}'.format(best_val_loss, best_val_f1, best_micro_val_f1))
        print('Test - Loss: {:.4f}, Macro_F1: {:.4f}, Micro_F1: {:.4f}'.format(best_test_loss, best_test_f1, best_micro_test_f1))
        final_f1.append(best_test_f1)
        final_micro_f1.append(best_micro_test_f1)

    print('--------------------Final Result-------------------------')
    print('Test - Macro_F1: {:.4f}+{:.4f}, Micro_F1:{:.4f}+{:.4f}'.format(np.mean(final_f1), np.std(final_f1), np.mean(final_micro_f1), np.std(final_micro_f1)))

Overwriting main_gpu.py


In [ ]:
import torch
import numpy as np
import pickle
import os
from scipy.sparse import csr_matrix
from torch_geometric.datasets import DBLP

# 1. Tải tập dữ liệu DBLP
print("Đang tải dữ liệu DBLP...")
dataset = DBLP(root='./data/DBLP_raw')
data = dataset[0]

# 2. Tạo không gian ID chung cho tất cả các loại nút
# Gộp Author, Paper, Term, Conference vào chung một ma trận kề duy nhất
node_types = ['author', 'paper', 'term', 'conference']
offsets = {}
current_offset = 0

for ntype in node_types:
    offsets[ntype] = current_offset
    current_offset += data[ntype].num_nodes
total_nodes = current_offset

# 3. Xây dựng Ma trận Kề (Adjacency Matrices)
edges = []
# Các loại quan hệ trong đồ thị DBLP
edge_types = [
    ('author', 'to', 'paper'),
    ('paper', 'to', 'author'),
    ('paper', 'to', 'term'),
    ('term', 'to', 'paper'),
    ('paper', 'to', 'conference'),
    ('conference', 'to', 'paper')
]

for etype in edge_types:
    if etype in data.edge_types:
        src_type, _, dst_type = etype
        edge_index = data[etype].edge_index.numpy()

        # Chuyển đổi ID cục bộ của từng loại nút sang ID chung
        src_indices = edge_index[0] + offsets[src_type]
        dst_indices = edge_index[1] + offsets[dst_type]

        # Khởi tạo giá trị 1 cho các cạnh có tồn tại
        vals = np.ones_like(src_indices)

        # Tạo ma trận thưa CSR
        adj = csr_matrix((vals, (src_indices, dst_indices)), shape=(total_nodes, total_nodes))
        edges.append(adj)

# 4. Tạo Đặc trưng Nút (Node Features)
# Các nút có số chiều đặc trưng khác nhau (Author: 334, Paper: 4231, Term: 50).
# Ta sẽ pad bằng 0 để tất cả có chung kích thước (w_in).
max_dim = max([data[ntype].x.shape[1] if 'x' in data[ntype] else 0 for ntype in node_types])
node_features = np.zeros((total_nodes, max_dim))

for ntype in node_types:
    start_idx = offsets[ntype]
    end_idx = start_idx + data[ntype].num_nodes
    if 'x' in data[ntype]:
        feat = data[ntype].x.numpy()
        dim = feat.shape[1]
        node_features[start_idx:end_idx, :dim] = feat

# 5. Xử lý Nhãn (Labels) cho tác vụ phân loại Author
author_labels = data['author'].y.numpy()
train_mask = data['author'].train_mask.numpy()
val_mask = data['author'].val_mask.numpy()
test_mask = data['author'].test_mask.numpy()

author_offset = offsets['author']

train_idx = np.where(train_mask)[0] + author_offset
valid_idx = np.where(val_mask)[0] + author_offset
test_idx = np.where(test_mask)[0] + author_offset

# Định dạng nhãn: [node_id, class_label]
train_label = np.vstack((train_idx, author_labels[train_mask])).T.astype(int)
valid_label = np.vstack((valid_idx, author_labels[val_mask])).T.astype(int)
test_label = np.vstack((test_idx, author_labels[test_mask])).T.astype(int)

labels = [train_label, valid_label, test_label]

# 6. Lưu file vào thư mục data/DBLP
output_dir = 'data/DBLP'
os.makedirs(output_dir, exist_ok=True)

with open(os.path.join(output_dir, 'node_features.pkl'), 'wb') as f:
    pickle.dump(node_features, f)
with open(os.path.join(output_dir, 'edges.pkl'), 'wb') as f:
    pickle.dump(edges, f)
with open(os.path.join(output_dir, 'labels.pkl'), 'wb') as f:
    pickle.dump(labels, f)

print(f"Xử lý thành công!")
print(f"Tổng số nút: {total_nodes}")
print(f"Số chiều đặc trưng (w_in): {max_dim}")
print(f"Dữ liệu đã được lưu tại: {output_dir}/")

Đang tải dữ liệu DBLP...
Xử lý thành công!
Tổng số nút: 26128
Số chiều đặc trưng (w_in): 4231
Dữ liệu đã được lưu tại: data/DBLP/


In [ ]:
!python main_gpu.py --dataset DBLP --model FastGTN --num_layers 3 --epoch 200 --lr 0.05 --channel_agg mean --num_channels 2

Namespace(model='FastGTN', dataset='DBLP', epoch=200, node_dim=64, num_channels=2, lr=0.05, weight_decay=0.001, num_layers=3, runs=10, channel_agg='mean', remove_self_loops=False, non_local=False, non_local_weight=0, beta=0, K=1, pre_train=False, num_FastGTN_layers=1)
Run 0
--------------------Best Result-------------------------
Train - Loss: 0.5726, Macro_F1: 0.9293, Micro_F1: 0.9300
Valid - Loss: 0.6135, Macro_F1: 0.7896, Micro_F1: 0.7950
Test - Loss: 0.5726, Macro_F1: 0.7987, Micro_F1: 0.8093
Run 1
--------------------Best Result-------------------------
Train - Loss: 0.5849, Macro_F1: 0.9246, Micro_F1: 0.9250
Valid - Loss: 0.6370, Macro_F1: 0.7940, Micro_F1: 0.7975
Test - Loss: 0.5849, Macro_F1: 0.7969, Micro_F1: 0.8066
Run 2
--------------------Best Result-------------------------
Train - Loss: 0.5983, Macro_F1: 0.9132, Micro_F1: 0.9150
Valid - Loss: 0.6386, Macro_F1: 0.7895, Micro_F1: 0.7925
Test - Loss: 0.5983, Macro_F1: 0.7976, Micro_F1: 0.8075
Run 3
--------------------Best R

In [ ]:
!python main_gpu.py --dataset DBLP --model GTN --num_layers 1 --epoch 100 --lr 0.02 --num_channels 1 --node_dim 32

Namespace(model='GTN', dataset='DBLP', epoch=100, node_dim=32, num_channels=1, lr=0.02, weight_decay=0.001, num_layers=1, runs=10, channel_agg='concat', remove_self_loops=False, non_local=False, non_local_weight=0, beta=0, K=1, pre_train=False, num_FastGTN_layers=1)
Traceback (most recent call last):
  File "/content/main_gpu.py", line 169, in <module>
    loss.backward()
  File "/usr/local/lib/python3.12/dist-packages/torch/_tensor.py", line 630, in backward
    torch.autograd.backward(
  File "/usr/local/lib/python3.12/dist-packages/torch/autograd/__init__.py", line 364, in backward
    _engine_run_backward(
  File "/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py", line 865, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
torch.OutOfMemoryError: CUDA out of memory. Tried to allocate 